# Generate COMPASS platinum-arm figures

Single source for all COMPASS grant figures. **Run All** renders the six
ARPI-restricted cohort arms: ICD-C61, VTE-project prostate, and their union,
each with and without the non-prostate-primary exclusion.

Outputs are organized by figure, then panel, for direct cohort comparison:

`FIG_ROOT/<figure>/<plot-stem>/python/<cohort>_<plot-stem>.png`

For example, `figure1/figure1a_consort/python/` contains the Figure 1A PNG
from all six cohorts. Plot PDFs are not generated. Tables remain CSV/Markdown
and use the same figure/panel/cohort-prefix convention.

All figure-generation logic lives in `COMPASS_generate_figures_pipeline.py`
as `generate_figures(cohort, ...)`. This notebook just imports that module
and calls it once per cohort, in-process, from a single cell.


## Setup


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "survival_common").exists()
)
sys.path.insert(0, str(REPO_ROOT / "COMPASS" / "survival_analysis"))

from COMPASS_generate_figures_pipeline import COHORTS, generate_figures

NEPC_PROJ_PATH = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS")
FIG_ROOT = Path("/data/gusev/USERS/jpconnor/figures/CAIA/COMPASS")


## Render all cohorts

Runs the full figure-generation pipeline once per cohort, in the same
kernel. Each call is self-contained (loads its own inputs, writes its own
cohort-prefixed outputs); a failure on one cohort is reported with its
cohort name rather than aborting the whole run silently.


In [ ]:
failures = []
for cohort in COHORTS:
    print(f"\n========== figures: {cohort} ==========")
    try:
        generate_figures(cohort, nepc_proj_path=NEPC_PROJ_PATH, fig_root=FIG_ROOT)
    except Exception as exc:
        print(f"FAILED {cohort}: {exc}")
        failures.append(cohort)
    else:
        print(f"completed {cohort}")

if failures:
    raise RuntimeError(f"Figure generation failed for cohorts: {failures}")
print("\nAll cohort figure sets completed.")
